# Stage 1: sodium and calcium experts

Train the initial Na and Ca experts, shared router, and saturation gate. The notebook calls the shared model implementation and reports metal readout and spectral-background errors.

`spice_moe.py` is required but absent from the supplied archive. The code below is preserved as the version-6 experiment definition.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["MOE_PROJECT_ROOT"]) if "MOE_PROJECT_ROOT" in os.environ else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from project_paths import configure_run
DATA_ROOT, RUN_ROOT, PREPARED_DIR, CHECKPOINT_DIR = configure_run(new=False)


In [ ]:
# === Step 0: environment, data plan, read-out indices ====================
import os, sys, time
import numpy as np, tensorflow as tf
sys.path.insert(0, os.getcwd())
import spice_moe as S

DATA_DIR = str(DATA_ROOT)
H5_DIR   = str(PREPARED_DIR)
CKPT_DIR = str(CHECKPOINT_DIR)
PLOT_DIR = str(RUN_ROOT / "figures")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PLOT_DIR, exist_ok=True)
np.random.seed(S.SEED); tf.random.set_seed(S.SEED)

# Rehearsal: every stage trains on ALL data seen so far, to reduce forgetting of
# an earlier interferent while learning the new one.
PLANS = ["na_train", "ca_train", "naca_train"]
P  = S.merge_plans([S.load_plan(H5_DIR, n) for n in PLANS])
wl = P["wl"]; SPEC_DIM = int(P["y_bank"].shape[1])
ridx  = S.build_readout_index(wl)      # the 4 wavelengths per metal the calibration reads
bmask = S.background_mask(wl)          # 1 outside the metal bands  -> the background
aw    = S.anchor_weights(wl, ridx)     # extra weight on those 4 wavelengths
tr_idx, va_idx = S.split_plan(P)       # whole (Cu,Ni,Zn) combinations held out
print("SPEC_DIM =", SPEC_DIM, " background wavelengths:", int(bmask.sum()), "/", len(bmask))


In [ ]:
# === Step 1: train stage1-NaCa ==============================================
# Stage 1 trains the Na and Ca experts, the shared router and the saturation
# gate from scratch.  With the gate fixed at 1.0 this would be a plain additive
# MoE; the gate is free, so it can learn that two ions together suppress less
# than the sum of their separate effects.
# Epochs / patience / batch are read from run_config.json, which 00_run_all writes,
# so all three stages can be tuned from one place.  Running this notebook on its own
# falls back to the built-in defaults in spice_moe.DEFAULT_RUN_CONFIG.
EPOCHS, PATIENCE, BATCH = S.stage_settings("stage1")
N_PAIR = int(BATCH * S.PAIR_FRACTION)     # rows that get a 2nd clean replicate

parts, model = S.build_stage(SPEC_DIM, ["Na", "Ca"], CKPT_DIR, None)
teacher = None
step_fn, opt, eval_fn = S.make_train_step(
    model, parts, ridx, P["bands"], bmask, aw, SPEC_DIM,
    lr=1e-3, total_steps=EPOCHS * S.STEPS_PER_EPOCH,
    train_ions=None, teacher=teacher, teacher_new_slots=None,
    n_pair=N_PAIR)

hist = S.fit_loop(step_fn, P, EPOCHS, batch=BATCH, label="stage1-NaCa",
                  val_idx=va_idx, eval_fn=eval_fn, track_models=[model],
                  patience=PATIENCE, n_pair=N_PAIR, seed=S.SEED)

S.save_parts(parts, CKPT_DIR, "stage1")


In [ ]:
# === Step 2: validation report — metals AND background ===================
# This is the check that matters for v6: the predicted validation distribution has
# to match the label distribution for BOTH the metal read-outs and the background.
# "gain" is std(pred)/std(label): 1.000 indicates matched readout standard deviations, not a complete calibration check.
# Compare MAE with measured repeatability while assessing bias and test design; further fitting may
# fits measurement noise.
sub = va_idx[:4000]
cond, pred, label = S.predict_on_plan(model, P, sub)
ro, bg = S.full_report(cond, pred, label, ridx, bmask, title="stage1-NaCa validation")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for k in ("full_rel", "anchor_rel", "contin_rel", "bkg_dist"):
    if k in hist: ax[0].plot(hist[k], label=k)
ax[0].set_yscale("log"); ax[0].set_title("background / relative terms"); ax[0].legend(fontsize=8)
for k in ("readout_hinge", "readout_bias", "readout_gain", "consistency"):
    if k in hist: ax[1].plot(hist[k], label=k)
ax[1].set_yscale("log"); ax[1].set_title("read-out terms"); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "stage1_loss.png"), dpi=120); plt.show()
